# 차원변환

In [4]:
import torch
tensor = torch.rand(2,4,3)
print(tensor)
print(tensor.shape)



tensor([[[0.3941, 0.3492, 0.2290],
         [0.2797, 0.3926, 0.6777],
         [0.3251, 0.7826, 0.1808],
         [0.9054, 0.8132, 0.1942]],

        [[0.2623, 0.7055, 0.4292],
         [0.4808, 0.8359, 0.3795],
         [0.3835, 0.6522, 0.4029],
         [0.8616, 0.4489, 0.6469]]])
torch.Size([2, 4, 3])


# 자료형 설정

In [5]:
tensor = torch.rand((2,4,3), dtype=torch.float32)
print(tensor)
print(tensor.shape)

tensor([[[0.2662, 0.5285, 0.8048],
         [0.6160, 0.5134, 0.0169],
         [0.5017, 0.4504, 0.7440],
         [0.0668, 0.4169, 0.1612]],

        [[0.3979, 0.2761, 0.2174],
         [0.6308, 0.2172, 0.5332],
         [0.8711, 0.0800, 0.7139],
         [0.2052, 0.4909, 0.6035]]])
torch.Size([2, 4, 3])


# 장치설정

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cpu = torch.FloatTensor([1.0, 2.0, 3.0])
gpu = torch.cuda.FloatTensor([1.0, 2.0, 3.0]) if torch.cuda.is_available() else None

tensor = torch.rand((1,1), device=device)
print(device)
print(cpu)
print(gpu)
print(tensor)

cuda
tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.8505]], device='cuda:0')


C:\Users\4cat1\AppData\Local\Temp\ipykernel_2824\1033247671.py:3: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  gpu = torch.cuda.FloatTensor([1.0, 2.0, 3.0]) if torch.cuda.is_available() else None


# 자동 미분 (Autograd) 
네, 맞습니다. 우리가 수학 시간에 종이에 풀 때는 $x$값 없이 $y = x^2 \rightarrow y' = 2x$ 라는 **"도함수(식)" 자체**를 구하죠. 이걸 **심볼릭 미분(Symbolic Differentiation)**이라고 합니다.

하지만 PyTorch 같은 딥러닝 라이브러리는 **자동 미분(Automatic Differentiation)** 방식을 씁니다.

### 차이점: "식 구하기" vs "값 구하기"

1.  **심볼릭 미분 (수학 시간)**
    *   목표: $2x$ 라는 **공식**을 찾아내는 것.
    *   도구: `SymPy` 같은 라이브러리가 이걸 합니다.
    *   장점: 식을 알면 분석하기 좋음.
    *   단점: 식이 너무 복잡해지면(변수 1억 개...) 컴퓨터가 식 자체를 못 만듭니다.

2.  **자동 미분 (PyTorch/딥러닝)**
    *   목표: **"지금 $x$가 3일 때, 기울기 값이 얼마야?" (숫자)** 만 알아내는 것.
    *   방식: 공식을 유도하지 않고, 그래프를 따라가면서 **숫자만 계산**합니다.
    *   장점: 변수가 1억 개라도(GPT 같은 거대 모델) 빠르게 숫자만 톡톡 계산해냅니다.

### 그래서 PyTorch는 값이 꼭 필요합니다
PyTorch는 **"공식을 모릅니다."**
그저 "아까 제곱했으니까, 미분값에 2랑 입력값(3)을 곱해주면 되겠군" 하고 **순간적인 계산**만 수행합니다. 그래서 반드시 초기값(`3.0`)이 들어있어야 그 시점의 기울기를 구할 수 있는 것입니다.

만약 공식(식) 자체가 필요하다면 파이썬의 `sympy` 라이브러리를 써야 합니다.
네, 딱 그 차이입니다!

*   **사람/SymPy**: $x^2 \xrightarrow{\text{미분}} 2x$ (함수를 구함)
*   **PyTorch**: $x=3$일 때, 기울기 $6$ (특정 지점의 **숫자**만 구함)

### 왜 이렇게 할까요? (딥러닝의 관점)

딥러닝의 목적은 "완벽한 수식"을 찾는 게 아니라, **"오차를 줄이는 방향(기울기)"**만 알면 되기 때문입니다.

산 정상에서 내려올 때 **"이 산의 전체 등고선 지도(함수 $2x$)"**는 필요 없고, **"지금 내 발밑의 경사가 어디가 급한지(값 6)"**만 알면 한 발짝 내려갈 수 있는 것과 똑같습니다.

그래서 PyTorch는 **"지도(수식)"를 만드는 힘든 일은 포기하고, "나침반(기울기 값)"만 빠르게 확인하는 전략**을 취한 겁니다. 덕분에 파라미터가 수천억 개여도 계산이 가능한 거죠

```python
# 식 자체를 구하고 싶을 때 (SymPy 사용)
import sympy

x = sympy.symbols('x')
f = x**2 + 2
diff_f = sympy.diff(f, x) 

print(diff_f)  # 결과: 2*x (공식이 나옴!)
```

In [ ]:
import torch

# 1. 텐서 생성 (requires_grad=True : "이 변수의 족보를 추적해라"라는 뜻)
x = torch.tensor(3.0, requires_grad=True)

# 2. 식 계산 : y = x^2 + 2
# (x=3 이니까 y = 9 + 2 = 11 이 됨)
y = x**3 + 2

# 3. 미분 실행! (y를 x로 미분해라)
y.backward()

# 4. 결과 확인 (수학적으로 y' = 2x 이므로, x=3일 때 기울기는 6이어야 함)
print(f"x 값: {x.item()}")
print(f"y 값: {y.item()}")
print(f"x의 기울기 (미분값): {x.grad}") # 정답: 6.0

x 값: 3.0
y 값: 29.0
x의 기울기 (미분값): 27.0


# 핵심 포인트 2가지
x.grad.zero_(): PyTorch는 기울기를 계속 누적(Accumulate)하는 성질이 있어서, 매 스텝마다 0으로 비워줘야 제대로 된 방향을 잡습니다.
with torch.no_grad():: 가중치를 직접 수정할 때는 미분 추적을 꺼야 합니다. 안 그러면 "가중치 수정하는 과정"까지 미분하려고 들어서 에러가 납니다.

In [5]:
import torch

# 1. 초기값 설정 (x=3에서 시작, 목표는 x=0으로 가는 것)
x = torch.tensor(3.0, requires_grad=True)
learning_rate = 0.1

for i in range(20):
    # 2. 순전파 (Forward): 식 계산 (여기선 y 자체가 오차/Loss 역할)
    y = x**2 + 2
    
    # 3. 역전파 (Backward): 기울기 계산
    y.backward()
    
    # 4. 경사하강법 (Update): x값 수정
    # with torch.no_grad()는 "이 계산은 미분 족보에 넣지 마라"는 뜻
    with torch.no_grad():
        x -= learning_rate * x.grad
        
    # 5. 기울기 초기화 (중요!)
    # 이걸 안 하면 이전 루프의 기울기(6)에 계속 더해져서(6+...) 엉망이 됨
    x.grad.zero_()
    
    # 결과 출력
    print(f"Step {i+1}: x = {x.item():.4f}, y(Loss) = {y.item():.4f}")


Step 1: x = 2.4000, y(Loss) = 11.0000
Step 2: x = 1.9200, y(Loss) = 7.7600
Step 3: x = 1.5360, y(Loss) = 5.6864
Step 4: x = 1.2288, y(Loss) = 4.3593
Step 5: x = 0.9830, y(Loss) = 3.5099
Step 6: x = 0.7864, y(Loss) = 2.9664
Step 7: x = 0.6291, y(Loss) = 2.6185
Step 8: x = 0.5033, y(Loss) = 2.3958
Step 9: x = 0.4027, y(Loss) = 2.2533
Step 10: x = 0.3221, y(Loss) = 2.1621
Step 11: x = 0.2577, y(Loss) = 2.1038
Step 12: x = 0.2062, y(Loss) = 2.0664
Step 13: x = 0.1649, y(Loss) = 2.0425
Step 14: x = 0.1319, y(Loss) = 2.0272
Step 15: x = 0.1056, y(Loss) = 2.0174
Step 16: x = 0.0844, y(Loss) = 2.0111
Step 17: x = 0.0676, y(Loss) = 2.0071
Step 18: x = 0.0540, y(Loss) = 2.0046
Step 19: x = 0.0432, y(Loss) = 2.0029
Step 20: x = 0.0346, y(Loss) = 2.0019


#  학습데이터 x y를 만들어서 해보자

네, 맞습니다! 이제 진짜 머신러닝처럼 데이터($x_{train}, y_{train}$)를 주고, 그 관계를 찾아내는 선형 회귀(Linear Regression) 모델을 만들어보죠.
우리가 찾을 목표는 $y = 3x + 2$ 라는 공식입니다. (컴퓨터는 이걸 모르는 상태)
이 코드를 돌리면 처음에 0x + 0으로 멍청하게 시작했다가, 점점 오차를 줄여가며 2.99x + 2.01 같이 정답에 근접해가는 과정을 볼 수 있습니다. 이게 딥러닝 학습의 기본 골격입니다.

In [8]:
import torch

# 1. 데이터 생성 (정답: W=3, b=2)
# y = 3x + 2 식에 약간의 노이즈 추가
x_train = torch.FloatTensor([[1], [2], [3], [4], [5]])
y_train = torch.FloatTensor([[5], [8], [11], [14], [17]])  # 3*x + 2 값들

# 2. 모델 초기화 (W와 b를 랜덤으로 찍고 시작)
# requires_grad=True: 이 변수들을 학습(업데이트)하겠다는 뜻
W = torch.zeros(1, requires_grad=True)  # 기울기(Weight) 초기값 0
b = torch.zeros(1, requires_grad=True)  # 절편(Bias) 초기값 0

# 학습 설정
lr = 0.01  # 학습률

print(f"학습 전 예측: y = {W.item():.2f}x + {b.item():.2f}")

# 3. 학습 루프
for epoch in range(1001):
    # (1) 가설(Prediction) 계산: H(x) = Wx + b
    hypothesis = x_train * W + b
    
    # (2) 비용(Cost/Loss) 계산: 평균 제곱 오차 (MSE)
    cost = torch.mean((hypothesis - y_train) ** 2)
    
    # (3) 역전파: 기울기 계산
    cost.backward()
    
    # (4) 파라미터 업데이트 (경사하강법)
    with torch.no_grad():
        W -= lr * W.grad
        b -= lr * b.grad
        
        # (5) 기울기 초기화 (필수!)
        W.grad.zero_()
        b.grad.zero_()
    
    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}/1000 | Loss: {cost.item():.6f} | W: {W.item():.4f}, b: {b.item():.4f}')

print(f"학습 후 최종 결과: y = {W.item():.2f}x + {b.item():.2f}")
# 정답인 y = 3x + 2 와 거의 비슷해졌는지 확인해보세요!

학습 전 예측: y = 0.00x + 0.00
Epoch    0/1000 | Loss: 139.000000 | W: 0.7800, b: 0.2200
Epoch  100/1000 | Loss: 0.108995 | W: 3.2136, b: 1.2288
Epoch  200/1000 | Loss: 0.055366 | W: 3.1522, b: 1.4503
Epoch  300/1000 | Loss: 0.028124 | W: 3.1085, b: 1.6082
Epoch  400/1000 | Loss: 0.014286 | W: 3.0773, b: 1.7208
Epoch  500/1000 | Loss: 0.007257 | W: 3.0551, b: 1.8010
Epoch  600/1000 | Loss: 0.003686 | W: 3.0393, b: 1.8582
Epoch  700/1000 | Loss: 0.001872 | W: 3.0280, b: 1.8989
Epoch  800/1000 | Loss: 0.000951 | W: 3.0200, b: 1.9280
Epoch  900/1000 | Loss: 0.000483 | W: 3.0142, b: 1.9487
Epoch 1000/1000 | Loss: 0.000245 | W: 3.0101, b: 1.9634
학습 후 최종 결과: y = 3.01x + 1.96


맞습니다. 그 "한 번에 계산"해주는 기능(Autograd)이 없었다면 딥러닝의 발전은 불가능했을 겁니다.
지금은 변수가 $W, b$ 딱 두 개뿐이라 손으로 미분해도 할 만하지만, 요즘 나오는 AI 모델들은 변수(파라미터)가 수억 개가 넘거든요.
그걸 사람이 일일이 미분 공식을 유도해서 코드로 짠다고 상상해보세요... 끔찍하죠. PyTorch는 그걸 "계산 그래프(Computational Graph)"라는 지도를 그려서 알아서 추적해주기 때문에, 우리는 "모델 구조(가설 식)"를 어떻게 짤지에만 집중할 수 있는 겁니다.
팁: 더 편하게 쓰는 법 (Optimizer)
지금 코드에서 직접 하셨던 "업데이트 과정"도 사실 더 줄일 수 있습니다.
지금 방식 (수동 업데이트):
with torch.no_grad():    
W -= lr * W.grad  # 일일이 빼주기    
b -= lr * b.grad    W.grad.zero_()    # 일일이 초기화
실무 방식 (Optimizer 사용):
# "나는 SGD라는 방식(경사하강법)으로 W랑 b를 관리할래" 라고 비서 고용
optimizer = torch.optim.SGD([W, b], lr=0.01)# ... (학습 루프 안에서) ...
cost.backward()       # 기울기 계산 (이건 똑같음)
optimizer.step()      # "업데이트 해!" (알아서 뺌)
optimizer.zero_grad() # "기울기 비워!" (알아서 초기화)
나중에는 이렇게 Optimizer라는 도구에게 귀찮은 업데이트 작업을 다 맡기게 됩니다.
import torch.optim: 최적화 도구를 가져옵니다.
optimizer = ...: 어떤 변수([W, b])를 업데이트할지 미리 등록합니다.
zero_grad() 위치: 보통 루프 시작하자마자 "지난번 기록 지워!" 하고 시작하는 게 관례입니다.
step(): W = W - lr * grad 같은 수식을 대신 처리해줍니다.
코드가 훨씬 "사람이 읽기 좋은 형태(High-level)"가 되었죠?


**Optimizer(최적화 도구)**는 쉽게 말해 **"산에서 내려가는 방법을 결정하는 길잡이"**입니다.

우리가 산 정상(높은 오차)에서 골짜기(낮은 오차)로 내려가야 하는데, **"어떻게 내려갈래?"**를 결정하고 실행해줍니다.

크게 **두 가지 역할**을 합니다.

### 1. 업데이트 실행 (Action)
우리가 아까 수동으로 했던 이 귀찮은 계산을 대신 해줍니다.
*   **수동:** `W = W - 0.01 * W.grad` (내가 직접 뺄셈)
*   **Optimizer:** `optimizer.step()` (얘가 알아서 뺌)

### 2. 내려가는 방법론 결정 (Strategy) - **이게 핵심!**
단순히 "기울기대로 빼기"만 하는 게 아니라, 더 똑똑하게 내려가는 기술들을 씁니다.

*   **SGD (우리가 쓴 것)**:
    *   "그냥 눈앞에 보이는 경사대로만 우직하게 내려가자."
    *   단점: 너무 느리거나, 구덩이(Local Minima)에 빠지면 못 나옴.

*   **Momentum (관성)**:
    *   "내려오던 속도가 있으니까, 방향이 좀 바뀌어도 관성으로 밀고 나가자!"
    *   구덩이를 휙 지나칠 수 있음.

*   **Adam (요즘 제일 많이 씀)**:
    *   "경사가 가파른 곳은 보폭을 줄여서 조심조심, 완만한 곳은 보폭을 넓혀서 성큼성큼!"
    *   **스피드(Momentum)**와 **정교함(Adaptive LR)**을 다 갖춘 똑똑한 길잡이.

### 요약하자면
Optimizer는 **"주인님은 기울기(grad)만 알려주세요. 제가 알아서 가장 좋은 보폭과 방향으로 파라미터($W, b$)를 수정해 놓겠습니다"** 라고 하는 충실한 대리인입니다.

In [10]:
import torch
import torch.optim as optim

# 1. 데이터 준비
x_train = torch.FloatTensor([[1], [2], [3], [4], [5]])
y_train = torch.FloatTensor([[5], [8], [11], [14], [17]]) # y = 3x + 2

# 2. 모델 초기화 (변수 선언)
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 3. Optimizer 정의 (비서 고용)
# "W와 b를 0.01의 학습률로 관리해줘"
optimizer = optim.SGD([W, b], lr=0.01)

print(f"시작 전: y = {W.item():.2f}x + {b.item():.2f}")

# 4. 학습 루프
for epoch in range(1001):
    
    # (1) 가설 계산
    hypothesis = x_train * W + b
    
    # (2) 비용 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # (3) 학습 진행 (3단 콤보)
    optimizer.zero_grad()  # 기울기 초기화 (항상 맨 처음에!)
    cost.backward()        # 역전파 (기울기 계산)
    optimizer.step()       # 파라미터 업데이트 (W, b 수정)

    # 로그 출력
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}/1000 | Loss: {cost.item():.6f} | W: {W.item():.4f}, b: {b.item():.4f}')

print(f"최종 결과: y = {W.item():.2f}x + {b.item():.2f}")

시작 전: y = 0.00x + 0.00
Epoch    0/1000 | Loss: 139.000000 | W: 0.7800, b: 0.2200
Epoch  100/1000 | Loss: 0.108995 | W: 3.2136, b: 1.2288
Epoch  200/1000 | Loss: 0.055366 | W: 3.1522, b: 1.4503
Epoch  300/1000 | Loss: 0.028124 | W: 3.1085, b: 1.6082
Epoch  400/1000 | Loss: 0.014286 | W: 3.0773, b: 1.7208
Epoch  500/1000 | Loss: 0.007257 | W: 3.0551, b: 1.8010
Epoch  600/1000 | Loss: 0.003686 | W: 3.0393, b: 1.8582
Epoch  700/1000 | Loss: 0.001872 | W: 3.0280, b: 1.8989
Epoch  800/1000 | Loss: 0.000951 | W: 3.0200, b: 1.9280
Epoch  900/1000 | Loss: 0.000483 | W: 3.0142, b: 1.9487
Epoch 1000/1000 | Loss: 0.000245 | W: 3.0101, b: 1.9634
최종 결과: y = 3.01x + 1.96


In [11]:
import torch
import torch.optim as optim

# 1. 데이터 준비
x_train = torch.FloatTensor([[1], [2], [3], [4], [5]])
y_train = torch.FloatTensor([[5], [8], [11], [14], [17]]) # y = 3x + 2

# 2. 모델 초기화 (변수 선언)
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 3. Optimizer 정의 (비서 고용)
# "W와 b를 0.01의 학습률로 관리해줘"
optimizer = optim.Adam([W, b], lr=0.1)

print(f"시작 전: y = {W.item():.2f}x + {b.item():.2f}")

# 4. 학습 루프
for epoch in range(1001):
    
    # (1) 가설 계산
    hypothesis = x_train * W + b
    
    # (2) 비용 계산
    cost = torch.mean((hypothesis - y_train) ** 2)

    # (3) 학습 진행 (3단 콤보)
    optimizer.zero_grad()  # 기울기 초기화 (항상 맨 처음에!)
    cost.backward()        # 역전파 (기울기 계산)
    optimizer.step()       # 파라미터 업데이트 (W, b 수정)

    # 로그 출력
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}/1000 | Loss: {cost.item():.6f} | W: {W.item():.4f}, b: {b.item():.4f}')

print(f"최종 결과: y = {W.item():.2f}x + {b.item():.2f}")

시작 전: y = 0.00x + 0.00
Epoch    0/1000 | Loss: 139.000000 | W: 0.1000, b: 0.1000
Epoch  100/1000 | Loss: 0.071116 | W: 2.8160, b: 2.5977
Epoch  200/1000 | Loss: 0.023382 | W: 2.8986, b: 2.3544
Epoch  300/1000 | Loss: 0.005459 | W: 2.9511, b: 2.1710
Epoch  400/1000 | Loss: 0.000903 | W: 2.9802, b: 2.0694
Epoch  500/1000 | Loss: 0.000107 | W: 2.9932, b: 2.0239
Epoch  600/1000 | Loss: 0.000009 | W: 2.9980, b: 2.0069
Epoch  700/1000 | Loss: 0.000001 | W: 2.9995, b: 2.0017
Epoch  800/1000 | Loss: 0.000000 | W: 2.9999, b: 2.0003
Epoch  900/1000 | Loss: 0.000000 | W: 3.0000, b: 2.0001
Epoch 1000/1000 | Loss: 0.000000 | W: 3.0000, b: 2.0000
최종 결과: y = 3.00x + 2.00
